# CS164 Optimization Methods — Session 20: Quadratic Programming

**Source and provenance:** Reconstructed from the five-page submitted *CS164 Session 20.pdf*, rather than an original notebook export. All five PCW questions and both Python code cells appear in the PDF. Equations and explanations have been polished for readability; the matrix-form coefficient typo in the written response is corrected below. No separate in-class work was supplied.

**Topics:** LP versus QP, strict convexity and degeneracy, Hessian positive definiteness, matrix form, CVXPY, feasible-region contour plots, covariance matrices.

**Run:** Use a Python 3 kernel with the repo's requirements installed. From the repository root, run \`python src/quadratic_programming.py\` or \`python src/quadratic_programming.py --save-dir session20_figures\`. This notebook imports the companion script, keeping both runnable formats consistent.


## Q1 — Recap of the previous lesson

We worked through a linear optimization problem in which the trajectory of a point-mass vehicle was optimized. The previous lesson's full model is not present in this Session 20 PCW document, so this is a recap rather than an invented reconstruction.


## Q2 — What is a quadratic program?

An **LP** has a linear objective and linear constraints. A **QP** has a quadratic objective with affine equality and/or inequality constraints, conventionally written as

$$\min_x\;\frac12x^TQx+c^Tx+d\quad\text{subject to }Ax\le b.$$

In **Session 15**, we minimized a squared-error objective in a drink-mixture problem subject to linear constraints on drink amounts. Earlier unconstrained quadratic minimization, including conjugate-gradient examples, is another special case of a QP.


## Q3 — Unique minimizers and possible locations

For a symmetric quadratic matrix $Q$, the Hessian of $f(x)=\frac12x^TQx+c^Tx+d$ is $Q$. If $Q\succ0$, the objective is **strictly convex**. If two different feasible points had the same minimum, their midpoint would be feasible (the feasible set is convex) and would have an even lower objective value, a contradiction. Therefore **if a minimum exists, it is unique**. Existence is guaranteed for a nonempty compact feasible set; it also holds for this exercise's coercive positive-definite quadratic even though its feasible set is unbounded.

On a bounded polygonal feasible set, the optimum may be **in the interior** (no active constraints), **on an edge** (an active constraint), or **at a vertex** (multiple active constraints). Unlike LP, QP solutions do not have to lie at vertices.

**Degenerate example:** $f(x_1,x_2)=x_1^2$ has Hessian $\operatorname{diag}(2,0)$, which is positive semidefinite but not definite. Over $[-1,1]^2$, all points $(0,x_2)$ are optimal. A singular Hessian allows multiple minimizers but does not imply them under every set of constraints. Here *non-degenerate* is taken to mean strictly convex over the feasible set.


## Q4 — Convexity and matrix form

Minimize

$$f(x_1,x_2)=3x_1^2+x_2^2+2x_1x_2+x_1+6x_2+2$$

subject to $2x_1+3x_2\ge4$, $x_1\ge0$, and $x_2\ge0$.

The gradient is $\nabla f=(6x_1+2x_2+1,\;2x_1+2x_2+6)^T$; the Hessian is

$$H=\begin{bmatrix}6&2\\2&2\end{bmatrix}.$$

Because its leading principal minors are $6>0$ and $\det(H)=8>0$, the Hessian is **positive definite**. With linear constraints, this is a strictly convex QP.

In $\min_x\frac12x^TQx+c^Tx+d$ subject to $Ax\le b$, use

$$
Q=\begin{bmatrix}6&2\\2&2\end{bmatrix},\qquad
c=\begin{bmatrix}1\\6\end{bmatrix},\qquad d=2,\qquad
A=\begin{bmatrix}-2&-3\\-1&0\\0&-1\end{bmatrix},\quad
b=\begin{bmatrix}-4\\0\\0\end{bmatrix}.
$$

**Correction:** The submitted text writes \`c = 1/6\`, but the correct linear coefficient vector is $c=(1,6)^T$, as used in the submitted code.


In [ ]:
# Imports and reusable implementation; launch from repository root or notebooks/.
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

root = Path.cwd()
if not (root / "src" / "quadratic_programming.py").exists():
    root = root.parent
if not (root / "src" / "quadratic_programming.py").exists():
    raise FileNotFoundError("Run from the CS164 repository or notebooks/ directory")
sys.path.insert(0, str(root))

from src.quadratic_programming import (
    Q, C, A, B, objective, plot_problem, solve_quadratic_program
)
print("Hessian Q:\n", Q)
print("Linear coefficients:", C)
print("Constraint matrix A:\n", A)
print("Bounds b:", B)
print("Hessian eigenvalues:", np.linalg.eigvalsh(Q))


In [ ]:
# Q4.3 — Contours and feasible region.
fig = plot_problem()
plt.show()


In [ ]:
# Q4.4 — CVXPY solution and optimal-point plot.
result = solve_quadratic_program()
print("Status:", result["status"])
print("Optimal solution [x1, x2]:", np.round(result["x"], 6))
print("Minimum objective value:", round(result["value"], 6))
print("Constraint slack b - Ax:", np.round(result["slack"], 6))
print("Active constraints:", result["active_constraints"])
fig = plot_problem(result["x"])
plt.show()


### Q4 — Numerical result and interpretation

The solver gives $x^*=(0.5,1)$ and $f(x^*)=11.25$. The line $2x_1+3x_2=4$ is active because $2(0.5)+3(1)=4$, whereas $x_1\ge0$ and $x_2\ge0$ are not active. The feasible region is **unbounded**, though the plot is a finite window.

An independent check along the active boundary, substituting $x_2=(4-2x_1)/3$, gives

$$f\left(x_1,\frac{4-2x_1}{3}\right)=\frac{19}{9}x_1^2-\frac{19}{9}x_1+\frac{106}{9},$$

which is minimized at $x_1=1/2$ and hence $x_2=1$. Since the constrained objective is strictly convex, this solution is unique.


## Q5 — Covariance matrix

A **covariance matrix** is a square matrix describing the variation of several variables individually and in relation to one another. The diagonal entries are variances; off-diagonal entries are pairwise covariances, indicating whether variables tend to move together or in opposite directions.

For random vector $X$ with mean $\mu$, the covariance matrix is $\Sigma=\mathbb E[(X-\mu)(X-\mu)^T]$. It is symmetric and **positive semidefinite**, since $v^T\Sigma v=\operatorname{Var}(v^TX)\ge0$ for every vector $v$. This connects covariance to the matrix-definiteness tests used in convex quadratic programming.
